# Testing the effect of varied source side anisotropy on receiver side anisotropy measurements

Some background: Shear wave splitting studies mainly use SKS phases as the wave of interest, mainly due to the fact that anisotropy measured can only come from the receiver side of the raypath (from some part of the mantle, but we assume that it is maily attributed to the upper mantle). SKS phases have their source side anisotropy erased as they transit through the outer core, and are thus an important phase to measure. Direct shear waves are trickier to use as data for these kinds of studies since they also contain information from source side splitting, when they initially propagate into the upper mantle. This combination of source and receiver side anisotropy with direct shear waves makes them harder to use when only trying to measure receiver side anisotropy.

However, the distribution of earthquakes across the surface of the Earth are not spread homogenously, and so certain regions of the globe do not have great SKS phase coverage. Seismic stations requires large events (mag5.5 - mag 7) to occur at epicentral distances of 90-120 degrees away from the station, and this can be seen in Chevrot's paper (2000) with maps of normalized coverage of these phases. This is prominent in the southwest Pacific, the study of interest. 

Looking at his maps, we see that the southwest Pacific has great coverage if we were to analyze direct S phases. It would be highly beneficial if we could utilize these phases in our Splitting Intensity workflow, as they would provide great azimuthal coverage for seismic stations to analyze receiver side azimuthal anisotropy. Is it possible that assumptions or alterations to our measurement process allows us to use these phases? 

I would like to test a number of things: 
1. How important is the incident angle of these direct S phase events when measuring anisotropy? If we assume a horizontal layer with no tilt, do the expressions presented in Romanowicz and Yuan 2012 mathematically justify the usage of direct S phases right alongside SKS phases? 
2. Direct S phases can come from a lot of different sources within the SWP: we have deep events, shallow events, events fro varied epicentral distances, events in exotic tectonic regions, events from different focal mechanisms...etc. Is there a golden standard of direct S phase from events that minimize source side anisotropy? maximize SNR? events or locations that help best with anisotropy? 
3. How does polarization azimuth help with this? need to review the math for this...
4. What is the tolerance for source side anisotropy when trying to meausre receiver side anisotropy? AKA, is there a maximum amount of source side that affects measurements? Should I measure this as a value of differentiated SI? What is the distribution of events per backazimuth that is acceptable to constrain results? Is this distribution gaussian or not? 
5. Should a bootstrapping method be implemented? This could help prove that variation in the back azimuth scatter and polarization direction of S event has less effect on the confidence of my SI than expected....can I create uncertainty bounds? Across both fast direction and delay time? 

# Create Catalog of Wavelets with SS and RS Anistropy 

use splitwavepy to create a large catalog of synthetic events from different back azimuths, fast directions, delay times, depths, noise, etc.
this catalog should be able to be queried in order to apply filters if need be...

In [1]:
# %matplotlib widget
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import itertools 
import splitwavepy as sw 
from scipy import signal
import sys
from support import SplittingIntensity, covar_particle_motion, envelopes
from obspy.signal.trigger import classic_sta_lta
import obspy
from obspy.core.stream import Stream 
from obspy.core.trace import Trace
import copy
import scipy.optimize as sci 

# from support import covar_particle_motion, envelopes


ModuleNotFoundError: No module named 'matplotlib.pyplot'

In [ ]:

# param_grid = {
#                 'fast_direction'      : np.around(np.linspace(0,360,45),2), 
#                 'delay_time'        : np.around(np.linspace(0,2,10),2),
#                 'noise'             :np.around(np.logspace(-2.5,0.5,10),4),
#                 'noisewidth'        :np.around(np.linspace(10,80,5),1),
#                 'width'             :np.around(np.linspace(20,160,5),1),
#                 'back_azimuth'      :np.around(np.linspace(0,360,21),2)

                
#                     }

param_grid = {
                'fast_direction'      : np.around(np.linspace(0,360,45),2), 
                'delay_time'        : np.around(np.linspace(0,3,10),2),
                'noise'             :np.around(np.logspace(-1.5,0,11),4),
                'noisewidth'        :np.around(np.linspace(20,100,6),1),     # 20 = highest freq noise, 100 = lowest freq noise, looks good from synthetic plots 
                'back_azimuth'      :np.around(np.linspace(0,360,61),2)

                
                    }
param_combination = list(itertools.product(*param_grid.values()))

# data = np.zeros((len(param_combination),8))



In [ ]:
len(param_combination)

np.logspace(-1.5,-0.2,11)

In [ ]:
len(param_combination)
param_combination


'''
Some Information Regarding what goes into the synthetics
split = [(fast1,dt1),(fast2,dt2)] splitting parameters applied to Ricker Wavelet IN ORDER OF ENTRY 
pol = polarization direction of wavelet in N/E coordinates, aka the back azimuth of the synthetic
delta = sampling rate, should match sample rate of real seismometers (20Hz I believe?)
nsamps = number of samples, should be equal to traces downloaded from real seismometers, 160 seconds I believe

width = width parameter of ricker wavelet, needs to match real data, 150 at 0.02 delta...      45 at 0.05 for 10s dominant period

noise = amplitude of gaussian noise applied
noisewidth = smoothing factor of gaussian noise applied 

'''

wavelet = sw.Pair(split=[(30,1.0)], 
                  pol=10, 

                  delta=0.05,
                  nsamps=3201,


                  width=45,
                  


                  noise=0.3, 
                  noisewidth=60)

plt.figure(figsize=(12,6))
plt.plot(wavelet.t(),wavelet.x,'b-',linewidth=1,alpha=0.7,label='Radial')
plt.plot(wavelet.t(),wavelet.y,'r-',linewidth=1,alpha=0.7,label='Transverse')
plt.legend()
plt.title('Synthetic Wavelet w/ Splitting')
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [unitless]")
# plt.xlim(70,90)
plt.show()


fftx = np.fft.rfft(wavelet.x)
ffty = np.fft.rfft(wavelet.y)

freqs = np.fft.rfftfreq(len(wavelet.x),d=wavelet.delta) 

plt.figure(figsize=(12,6))
plt.semilogx(freqs, np.abs(fftx), 'b-', linewidth=1, alpha=0.7, label='Radial')
plt.semilogx(freqs, np.abs(ffty), 'b-', linewidth=1, alpha=0.7, label='Radial')

plt.title('FFT of Synthetic Wavelet (Radial Component)')
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude [unitless]")
# plt.xlim(0,1)
# plt.xticks(np.linspace(0,1,11))
plt.show()



# wavelet.plot()



In [ ]:
# Test different width values
for w in [30,40, 45, 60]:
    test_wavelet = sw.Pair(split=[(30,1.0)], pol=10, delta=0.05, nsamps=3201, width=w, noise=0.0, noisewidth=100)
    fft_test = np.fft.rfft(test_wavelet.x)
    freqs = np.fft.rfftfreq(len(test_wavelet.x), d=test_wavelet.delta)
    peak_freq = freqs[np.argmax(np.abs(fft_test))]
    print(f"Width={w}: Peak frequency = {peak_freq:.3f} Hz (Period = {1/peak_freq:.1f}s)")

In [ ]:
# Make a ton of these
# 'fast_direction'      : np.around(np.linspace(0,360,45),2), 
#                 'delay_time'        : np.around(np.linspace(0,2,10),2),
#                 'noise'             :np.around(np.logspace(-2.5,0.5,10),4),
#                 'noisewidth'        :np.around(np.linspace(10,80,5),1),
#                 'width'             :np.around(np.linspace(20,160,5),1),
#                 'back_azimuth'      :np.around(np.linspace(0,360,21),2)
data = np.zeros((len(param_combination),11))

'''
Create a wavelet pair using splitwavepy using a combination of parameters from list. 
Then create obspy stream object with traces being the x and y data of the wavelet.
Apply routine signal processing as applied to real data. 
Save everything to array after, process as dataframe, then filter according to current standards. 
This should enable me to see proper filtering parameters with a wide array of noisy synthetic data. 

'''

# plt.figure(figsize=(40,20))
# # plt.subplot(2,4,1)
# ticker = 0 


for i in range(0,len(param_combination),2000):
    fast,dt,noise,noisewidth,baz = param_combination[i]
    # print('yo')
    wavelet = sw.Pair(  split=[(fast,dt),(45,1)], 
                        pol=baz, 
                        delta=0.05,
                        nsamps=3201,
                        width=45,
                        noise=noise, 
                        noisewidth=noisewidth
                        )   
    
    # Create trace objects to store data
    traceX = Trace(data=wavelet.x)
    traceY = Trace(data=wavelet.y)
    traceX.stats.sampling_rate , traceX.stats.channel = 20 , "X"
    traceY.stats.sampling_rate , traceY.stats.channel = 20 , "Y"

    # make waveform stream and copy for particle motion
    wf = Stream(traces=[traceX,traceY])
    particleMotion_wf = copy.deepcopy(wf)


    # Filtering
    freqmin, freqmax = 0.01, 0.125
    wf.detrend('demean')
    wf.detrend('linear')
    wf.filter("bandpass", freqmin=freqmin, freqmax=freqmax)

    north = wf.select(component="X")[0].data
    east  = wf.select(component="Y")[0].data
    sample_interval = wf[0].stats.delta

    if len(north) % 2 == 0: north = north[:-1]
    if len(east) % 2 == 0: east = east[:-1]

    n_sta_lta = classic_sta_lta(north,100,400)
    e_sta_lta = classic_sta_lta(east,100,400)

    time = list(range(len(wavelet.x)))
    sum_sta_lta = n_sta_lta+e_sta_lta
    phase_ind = np.argmax(sum_sta_lta)
    sta_lta_max = sum_sta_lta[phase_ind]
    sta_lta_max_time = time[phase_ind]*sample_interval
    
    # delta_travel = abs(sta_lta_max_time - 60)

    windowstart = phase_ind - 300 
    windowend = phase_ind + 300
    # print(windowstart,windowend)

    if windowend >= len(wavelet.x):
        continue


    # Particle motion
    particleMotion_wf.detrend('demean')
    particleMotion_wf.detrend('linear')
    particleMotion_wf.filter('bandpass', freqmin=0.02, freqmax=0.066)
    particleMotion_wf_north = particleMotion_wf.select(component="X")[0].data
    particleMotion_wf_east  = particleMotion_wf.select(component="Y")[0].data
    if len(particleMotion_wf_north) % 2 == 0: particleMotion_wf_north = particleMotion_wf_north[:-1]
    if len(particleMotion_wf_east) % 2 == 0: particleMotion_wf_east = particleMotion_wf_east[:-1]

    calcbaz, xlam1, xlam2 = covar_particle_motion(particleMotion_wf_north[windowstart:windowend],
                                                    particleMotion_wf_east[windowstart:windowend])

    if abs(calcbaz - baz) > 90:
        calcbaz = (calcbaz + 180) % 360


    # Envelope / SNR
    env_pair = sw.Pair(north, east, delta=sample_interval)
    mean_snr, max_snr, ne_env, mean_snr2, max_snr2 = envelopes(env_pair, baz, calcbaz, windowstart, windowend)

    pair = sw.Pair(north, east, delta=sample_interval)
    pair.set_window(float(windowstart*sample_interval), float(windowend*sample_interval))
    chopped_wf = pair.chop()

    # wiggleSIlow_act , wiggleSI_act , wiggleSIhigh_act = SplittingIntensity(wavelet,float(baz))
    wiggleSIlow_calc , wiggleSI_calc , wiggleSIhigh_calc = SplittingIntensity(chopped_wf,float(calcbaz))

    if abs(calcbaz - baz) > 90:
            calcbaz = (calcbaz+180) % 360
        # print(baz,fast1,dt1)

    data[i,:] = fast,dt,noise,noisewidth,45,baz,calcbaz,wiggleSI_calc,mean_snr,mean_snr2,xlam1/xlam2



    # if i % 200000 == 0: 
    #     ticker += 1 
    #     # row += 1 

    #     # plt.figure(figure=(10,5))
    #     plt.subplot(5,5,ticker)
    #     plt.plot(wavelet.t(),wavelet.x,'b-',linewidth=1,alpha=0.7,label='Radial')
    #     plt.plot(wavelet.t(),wavelet.y,'r-',linewidth=1,alpha=0.7,label='Transverse')

    #     if ticker == 25:
    #         plt.show()
         
    
    # print(baz,calcbaz)

In [ ]:
mask = data.any(axis=1)

data = data[mask]

len(data)

column_names = [
    'fast_direction',      # fast
    'delay_time',          # dt  
    'noise_level',         # noise
    'noise_width',         # noisewidth
    'wavelet_width',       # width
    'true_backazimuth',    # baz
    'calc_backazimuth',    # calcbaz
    'splitting_intensity', # wiggleSI_calc
    'mean_snr',           # mean_snr
    'mean_snr2',          # mean_snr2
    'eigenvalue_ratio'    # xlam1/xlam2
]

# Create DataFrame
entire_df = pd.DataFrame(data, columns=column_names)


In [ ]:
entire_df.head()

In [ ]:
df = entire_df
df = df[df['mean_snr'] > 1]
df = df[df['mean_snr2'] > 1]
df = df[abs(df['splitting_intensity'])<4]
# df = df[df['eigenvalue_ratio'] > 10]


print(f"Entire DF Length: {len(entire_df)}")
print(f"Filtered DF Length: {len(df)}")


event_baz = df['true_backazimuth']
polarization_azimuths = df['calc_backazimuth']
splitIntensity = df['splitting_intensity']

plt.figure(figsize=(12,8))
# plt.plot(event_baz,splitIntensity,'o',color='r',alpha=0.5)
plt.plot(polarization_azimuths,splitIntensity,'o',color='b',alpha=0.5)
# plt.plot(event_baz,splitIntensity,'o',alpha=0.7)
# plt.plot(event_baz,abs(polarization_azimuths-event_baz),'o')

plt.show()


In [ ]:
print(f"Filtered DF Length: {len(df)}")
selected_indices = np.random.choice(len(df), size=len(df) // 16 , replace=False)
df = df.iloc[selected_indices]
print(f"Random Selected DF Length: {len(df)}")



selected_indices = np.random.choice(len(df), size=len(df) // 10 , replace=False)
wavelet_df = df.iloc[selected_indices]
print(f"Random Selected Wavelet Plots Length: {len(wavelet_df)}")

In [ ]:
# Even simpler - just print them
for name, value in wavelet.__dict__.items():
    print(f"{name}: {sys.getsizeof(value)} bytes")

In [ ]:
# Quick and dirty version
def quick_bin_data(df, bin_width=10):
    """Quick binning without confidence intervals."""
    
    array = np.arange(0, 360, bin_width)
    df['chunk'] = np.digitize(df['calc_backazimuth'], array, right=True)
    
    # Group by bins and calculate statistics
    binned = df.groupby('chunk').agg({
        'splitting_intensity': ['mean', 'std', 'count'],
        'calc_backazimuth': 'mean'
    }).reset_index()
    
    # Flatten column names
    binned.columns = ['chunk', 'mean_SI', 'std_SI', 'count', 'mean_azimuth']
    
    # Add bin centers
    binned['bin_center'] = (binned['chunk'] - 1) * bin_width + bin_width/2
    
    # Filter minimum count
    binned = binned[binned['count'] >= 2]
    
    return binned

binned = quick_bin_data(df)

# binned

In [ ]:
# SI function for nonlinear least squares regression
def SI_function(baz,fast,lag):
    SI = lag * 2 * np.sin(np.deg2rad(baz-fast)) * np.cos(np.deg2rad(baz-fast))
    return SI
bounds = ([-360, 0], [360,4])


# sine_fit = sci.curve_fit(SI_function,binned_results['azimuth'],binned_results['mean'],sigma=binned_results['weight'],bounds=bounds,full_output=True,nan_policy='omit') #,p0=p0)
sine_fit = sci.curve_fit(SI_function,df['calc_backazimuth'],df['splitting_intensity'],bounds=bounds,full_output=True) #,p0=p0)

params,cov,fitdata,trash1,trash2 = sine_fit
phi,dt = params[0],params[1]

binned_sine_fit = sci.curve_fit(SI_function,binned['mean_azimuth'],binned['mean_SI'],sigma=binned['std_SI'],bounds=bounds,full_output=True) #,p0=p0)

params,cov,fitdata,trash1,trash2 = sine_fit
bin_phi,bin_dt = params[0],params[1]



In [ ]:
print(f"Fast Direction:{phi}, Delay Time: {dt}")

event_baz = df['true_backazimuth']
polarization_azimuths = df['calc_backazimuth']
splitIntensity = df['splitting_intensity']

degs = np.arange(0,360,1)

plt.figure(figsize=(12,8))
# plt.plot(event_baz,splitIntensity,'o',color='r',alpha=0.5)
plt.plot(polarization_azimuths,splitIntensity,'o',color='b',alpha=0.5)
# plt.plot(event_baz,splitIntensity,'o',alpha=0.7)
# plt.plot(event_baz,abs(polarization_azimuths-event_baz),'o')
plt.plot(degs,SI_function(degs,phi,dt),'r-',linewidth=3)
plt.plot(binned['mean_azimuth'],binned['mean_SI'],'o',color='red',markersize=10,alpha=0.8)
plt.errorbar(binned['mean_azimuth'],binned['mean_SI'],yerr=binned['std_SI'],xerr=None,ecolor='red',barsabove=True,alpha=0.5,linestyle='None')

# ax1.errorbar(eventdata['calcbaz'].iloc[i],eventdata['wiggleSI_calc'].iloc[i],yerr=eventdata['SI_calc_error'].iloc[i],xerr=None,ecolor='blue',barsabove=True,alpha=0.5)

plt.show()

# azimuthal slice of SI and look at distribution
plt.figure(figsize=(8,6))

slice = df[(df['calc_backazimuth']>140)&(df['calc_backazimuth']<160)]

plt.hist(slice['splitting_intensity'],bins=45)


plt.show()

In [ ]:



wavelet = sw.Pair(  split=[(fast,dt),(30,1)], 
                        pol=baz, 
                        delta=0.05,
                        nsamps=3201,
                        width=width,
                        noise=noise, 
                        noisewidth=noisewidth
                        )  